In [ ]:
import os
import json
import random
import pandas as pd
from tqdm import tqdm

from gemini_integration import load_api_keys, init_gemini, call_gemini

# Path to the prepared data
PARAGRAPH_PARQUET = "./scrape_and_prepare_data/haifa_prepared_data/haifa_paragraph_index_config_chunk1000_overlap200.parquet"

# Output testset file
OUT_CSV = "./evaluation/haifa_testset.csv"

NUM_EXAMPLES = 80          # Number of QA pairs to generate
MAX_CONTEXT_CHARS = 1200   # Limit context size for prompt

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# Load Gemini
api_keys = load_api_keys("api_keys.json")
gemini_model = init_gemini(api_keys, model_name="gemini-2.5-flash")


In [ ]:
# Load prepared data

df = pd.read_parquet(PARAGRAPH_PARQUET)
print("Total rows:", len(df))

# Optional filtering: keep only HTML chunks
if "file_type" in df.columns:
    df = df[df["file_type"] == "html"].copy()

# Keep relevant columns only
cols = ["doc_id", "url", "title", "subtitle", "text", "chunk_text_only"]
cols = [c for c in cols if c in df.columns]
df = df[cols].dropna(subset=["text"]).reset_index(drop=True)

print("Filtered rows:", len(df))
df.head()


In [ ]:
# Gemini helper for QA generation

def build_qa_generation_prompt(context: str, url: str = "", title: str = "") -> str:
    """
    Build a Gemini prompt that asks for a question-answer pair
    based strictly on the given municipal text.
    """
    meta = []
    if title:
        meta.append(f"כותרת המסמך: {title}")
    if url:
        meta.append(f"כתובת המסמך: {url}")
    meta_str = "\n".join(meta)

    prompt = f"""
להלן קטע מידע רשמי מאתר עיריית חיפה.

{meta_str}

טקסט:
\"\"\" 
{context}
\"\"\"

בהתבסס אך ורק על המידע הזה, צור זוג של שאלה ותשובה (Q&A) בעברית שמתאים לתושב חיפה.

הנחיות:
1. השאלה צריכה להיות טבעית, כמו שתושב ישאל (משפט אחד).
2. התשובה צריכה להיות נכונה, קצרה וברורה (1–4 משפטים), ולתאר את המידע בדייקנות.
3. אל תשתמש בידע חיצוני שלא מופיע בטקסט.
4. אל תמציא פרטים שלא רשומים בטקסט.

החזר תשובה בפורמט JSON תקין, ללא טקסט נוסף, במבנה הבא:

{{
  "question": "<השאלה בעברית>",
  "answer": "<התשובה בעברית>"
}}
"""
    return prompt.strip()


In [ ]:
# Generate a QA pair for a single prepared-data row
def generate_qa_for_row(row) -> dict:
    """
    Create a QA pair for a single prepared-data row.
    Uses chunk_text_only if available, else text.
    """
    text = row.get("chunk_text_only") or row.get("text") or ""
    text = str(text).strip()

    if len(text) > MAX_CONTEXT_CHARS:
        text = text[:MAX_CONTEXT_CHARS]

    if not text:
        return None

    prompt = build_qa_generation_prompt(
        context=text,
        url=row.get("url", ""),
        title=row.get("title", "")
    )

    try:
        resp_text = call_gemini(gemini_model, prompt)
        data = json.loads(resp_text)

        question = data.get("question", "").strip()
        answer = data.get("answer", "").strip()

        if not question or not answer:
            return None

        return {
            "question": question,
            "answer": answer,
            "doc_id": row.get("doc_id", ""),
            "url": row.get("url", ""),
            "title": row.get("title", ""),
        }

    except Exception as e:
        print("Failed to generate QA:", e)
        return None


In [ ]:
# Generate the full testset
examples = []
indices = list(range(len(df)))
random.shuffle(indices)

for idx in tqdm(indices, total=min(len(indices), NUM_EXAMPLES)):
    if len(examples) >= NUM_EXAMPLES:
        break

    row = df.iloc[idx]
    qa = generate_qa_for_row(row)
    if qa is not None:
        examples.append(qa)

test_df = pd.DataFrame(examples)
print("Generated test examples:", len(test_df))
test_df.head()

# Save the testset
test_df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("Saved testset to:", OUT_CSV)

